# `tsl` Development

This notebook is to test out the [Torch Spatiotemporal](https://torch-spatiotemporal.readthedocs.io/en/latest/usage/quickstart.html) package as an alternative to PyG. It's dataloaders and models are specifically structured for neural spatiotemporal data processing. It also uses Hydra for reproducible experiments.


## SpatioTemporalDataset

In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np
import sys
import os

import torch

from tsl.data import SpatioTemporalDataset

# # Add parent directory to path to import cgnn modules
from cgnn.process_data import (
    get_date_range,
    get_mobility_df_by_source,
    process_hospitalization_data,
    process_case_death_data,
    create_node_key,
    create_edge_index,
    process_advan_data,
    process_safegraph_data,
    create_train_test_mask,
)
from cgnn.utils import get_cbsa_list
from omegaconf import DictConfig, OmegaConf

from tqdm import tqdm

In [ ]:
def select_diverse_cbsas(n=15):
    """
    Select n diverse CBSAs across the country.
    Returns a list of CBSA codes.
    """
    all_cbsas = get_cbsa_list()
    
    # Select major metropolitan areas from different regions
    # This is a simple selection - you could make it more sophisticated
    # by using geographic coordinates or HHS regions
    selected = [
        "12060",  # Atlanta-Sandy Springs-Roswell, GA (South)
        "12420",  # Austin-Round Rock-San Marcos, TX (South)
        "12540",  # Bakersfield-Delano, CA (West)
        "12580",  # Baltimore-Columbia-Towson, MD (Northeast)
        "13820",  # Birmingham, AL (South)
        "14460",  # Boston-Cambridge-Newton, MA-NH (Northeast)
        "15380",  # Buffalo-Cheektowaga-Niagara Falls, NY (Northeast)
        "16740",  # Charlotte-Concord-Gastonia, NC-SC (South)
        "16980",  # Chicago-Naperville-Elgin, IL-IN-WI (Midwest)
        "19100",  # Dallas-Fort Worth-Arlington, TX (South)
        "19740",  # Denver-Aurora-Lakewood, CO (West)
        "19820",  # Detroit-Warren-Dearborn, MI (Midwest)
        "26420",  # Houston-The Woodlands-Sugar Land, TX (South)
        "31080",  # Los Angeles-Long Beach-Anaheim, CA (West)
        "35620",  # New York-Newark-Jersey City, NY-NJ-PA (Northeast)
    ]
    
    # Ensure all selected CBSAs are in the full list
    selected = [cbsa for cbsa in selected if cbsa in all_cbsas]
    
    # If we need more, add from the full list
    if len(selected) < n:
        remaining = [cbsa for cbsa in all_cbsas if cbsa not in selected]
        selected.extend(remaining[:n - len(selected)])
    
    return selected[:n]


In [ ]:
def create_tsl_spatiotemporal_data(
    num_cbsas=15,
    data_source="hospital",
    mobility_source="advan_plus",
    window=7,
    horizon=1,
    delay=0,
    stride=1,
    cfg=None,
):
    """
    Create a SpatioTemporalDataset in tsl format from processed data.
    
    Args:
        num_cbsas: Number of CBSAs to include
        data_source: "hospital" or "case"
        mobility_source: "advan" or "safegraph"
        window: Input sequence length for sliding window
        horizon: Forecast horizon
        delay: Steps between window end and target start
        stride: Steps between consecutive samples
        cfg: Optional Hydra config
        
    Returns:
        SpatioTemporalDataset: Dataset in tsl format
    """
    # Select CBSAs
    cbsa_list = get_cbsa_list()
    print(f"Selected {len(cbsa_list)} CBSAs: {cbsa_list[:5]}...")
    
    # Get date range - use config or defaults
    if cfg is not None and hasattr(cfg, "data"):
        start_date = cfg.data.get("start_date", "03/22/2020")
        end_date = cfg.data.get("end_date", "12/31/2020")
    else:
        start_date = "07/13/2020"
        end_date = "12/31/2021"
    
    all_dates = get_date_range(start_date, end_date)
    num_timesteps = len(all_dates)
    # Select first num_timesteps
    dates = all_dates[:num_timesteps]
    print(f"Selected {len(dates)} time steps: {dates[0]} to {dates[-1]}")
    
    # Create a minimal config for processing
    if cfg is None:
        cfg = OmegaConf.create({
            "data": {
                "start_date": start_date,
                "end_date": end_date,
                "data_source": data_source,
                "mobility_source": mobility_source,
            }
        })
    
    # Process target data (hospitalization or case data)
    if data_source == "hospital":
        hosp_df = process_hospitalization_data(cbsa_list, cfg=cfg)
        # Get hosp_col from config or default
        hosp_col = cfg.data.get("hosp_col", "total_adult_patients_hospitalized_confirmed_covid_7_day_sum")
        
        # Filter to selected dates
        hosp_df = hosp_df[hosp_df["collection_week"].isin(dates)]
        
        # Pivot to get [time, nodes, features] format
        target_df = hosp_df.pivot_table(
            index="collection_week",
            columns="CBSA",
            values=hosp_col,
            fill_value=0
        )
        
        # Ensure CBSAs are in the right order
        target_df = target_df.reindex(columns=cbsa_list, fill_value=0)
        target_df = target_df.reindex(index=dates, fill_value=0)
        
        # Convert to tensor: [time, nodes, features]
        target = torch.tensor(target_df.values, dtype=torch.float32)
        # Add feature dimension: [time, nodes, 1]
        target = target.unsqueeze(-1)
        
    else:  # case data
        death_df, case_df = process_case_death_data(cbsa_list, cfg=cfg)
        
        # Filter to selected dates
        case_df = case_df[case_df["date"].isin(dates)]
        death_df = death_df[death_df["date"].isin(dates)]
        
        # Merge case and death data
        merged_df = case_df.merge(death_df, on=["date", "CBSA"], how="outer", suffixes=("_case", "_death"))
        merged_df = merged_df.fillna(0)
        
        # Select feature columns
        feature_cols = [
            "CASE_COUNT",
            "CASE_COUNT_7DAY_AVG",
            "DEATH_COUNT",
            "DEATH_COUNT_7DAY_AVG",
        ]
        
        # Create multi-index pivot for each feature
        target_list = []
        for col in feature_cols:
            if col in merged_df.columns:
                pivot_df = merged_df.pivot_table(
                    index="date",
                    columns="CBSA",
                    values=col,
                    fill_value=0
                )
                pivot_df = pivot_df.reindex(columns=cbsa_list, fill_value=0)
                pivot_df = pivot_df.reindex(index=dates, fill_value=0)
                target_list.append(pivot_df.values)
        
        # Stack features: [time, nodes, features]
        target = torch.tensor(np.stack(target_list, axis=-1), dtype=torch.float32)
    
    print(f"Target shape: {target.shape}")  # Should be [time, nodes, features]
    
    # Process mobility data for edges
    mobility_df = get_mobility_df_by_source(mobility_source, cbsa_list, cfg=cfg)
    
    # Create node_dict for mapping (but we'll work with CBSA indices directly)
    # In tsl format, nodes are just CBSAs (spatial entities)
    cbsa_to_idx = {cbsa: idx for idx, cbsa in enumerate(cbsa_list)}
    
    # Create connectivity list: list of (edge_index, edge_weight) tuples for each time step
    edge_index_list = []
    edge_weight_list = []
    
    print("Processing edges for each time step...")
    # Convert date_range_start to datetime if it's not already
    if not pd.api.types.is_datetime64_any_dtype(mobility_df["date_range_start"]):
        mobility_df["date_range_start"] = pd.to_datetime(mobility_df["date_range_start"])
    
    # Create a set of date strings for faster lookup
    date_strs = set(date.strftime("%Y-%m-%d") for date in dates)
    
    for t, date in tqdm(enumerate(dates)):
        date_str = date.strftime("%Y-%m-%d")
        
        # Filter mobility data for this date
        date_mobility = mobility_df[
            mobility_df["date_range_start"].dt.strftime("%Y-%m-%d") == date_str
        ]
        
        # Create edge_index for this time step
        edge_list = []
        weight_list = []
        
        for _, row in date_mobility.iterrows():
            cbsa_orig = row["cbsa_orig"]
            cbsa_dest = row["cbsa_dest"]
            
            # Only include edges between selected CBSAs
            if cbsa_orig in cbsa_to_idx and cbsa_dest in cbsa_to_idx:
                orig_idx = cbsa_to_idx[cbsa_orig]
                dest_idx = cbsa_to_idx[cbsa_dest]
                edge_list.append([orig_idx, dest_idx])
                weight_list.append(row["visitor_home_aggregation"])
        
        if len(edge_list) > 0:
            edge_index_t = torch.tensor(edge_list, dtype=torch.long).t()  # [2, num_edges]
            edge_weight_t = torch.tensor(weight_list, dtype=torch.float32)
        else:
            # No edges for this time step - create empty tensors
            edge_index_t = torch.empty((2, 0), dtype=torch.long)
            edge_weight_t = torch.empty((0,), dtype=torch.float32)
        
        # Store as (edge_index, edge_weight) tuple for connectivity
        edge_index_list.append(edge_index_t)
        edge_weight_list.append(edge_weight_t)
    
    # Create SpatioTemporalDataset with connectivity parameter
    dataset = SpatioTemporalDataset(
        target=target,
        connectivity=None,
        window=window,
        horizon=horizon,
        delay=delay,
        stride=stride,
    )
    
    dataset.edge_index = edge_index_list
    dataset.edge_weight = edge_weight_list
    
    return dataset, cbsa_list, dates


In [ ]:
# Create the dataset
dataset, cbsa_list, dates = create_tsl_spatiotemporal_data(
    num_cbsas=15,
    data_source="hospital",
    mobility_source="advan_plus",
    window=7,
    horizon=1,
    delay=0,
    stride=1,
)

print(f"\nDataset created successfully!")
print(f"Number of samples: {len(dataset)}")
print(f"Dataset window: {dataset.window}")
print(f"Dataset horizon: {dataset.horizon}")


In [ ]:
# Inspect a sample from the dataset
sample = dataset[0]

print("Sample structure:")
print(f"  x (input): shape {sample.x.shape}, dtype {sample.x.dtype}")
print(f"  y (target): shape {sample.y.shape}, dtype {sample.y.dtype}")

# Check if edge_index and edge_weight are in the sample
if hasattr(sample, 'edge_index'):
    if isinstance(sample.edge_index, list):
        print(f"  edge_index: list of {len(sample.edge_index)} tensors")
        if len(sample.edge_index) > 0:
            print(f"    First edge_index shape: {sample.edge_index[0].shape}")
    else:
        print(f"  edge_index: shape {sample.edge_index.shape}")

if hasattr(sample, 'edge_weight'):
    if isinstance(sample.edge_weight, list):
        print(f"  edge_weight: list of {len(sample.edge_weight)} tensors")
        if len(sample.edge_weight) > 0:
            print(f"    First edge_weight shape: {sample.edge_weight[0].shape}")
    else:
        print(f"  edge_weight: shape {sample.edge_weight.shape}")

# Print all attributes
print(f"\nAll sample attributes: {list(sample.keys())}")


In [ ]:
# Visualize some statistics
print("Dataset Statistics:")
print(f"  Original target shape: {dataset.target.shape}")
print(f"  Number of CBSAs: {dataset.target.shape[1]}")
print(f"  Number of time steps: {dataset.target.shape[0]}")
print(f"  Number of features: {dataset.target.shape[2]}")

# Check edge statistics
edge_counts = [ei.shape[1] for ei in dataset.edge_index if ei.shape[1] > 0]
if edge_counts:
    print(f"\nEdge Statistics:")
    print(f"  Min edges per time step: {min(edge_counts)}")
    print(f"  Max edges per time step: {max(edge_counts)}")
    print(f"  Mean edges per time step: {np.mean(edge_counts):.1f}")
    print(f"  Total unique edges across all time steps: {sum(edge_counts)}")

# Show first few CBSAs
print(f"\nFirst 5 CBSAs: {cbsa_list[:5]}")
print(f"First 5 dates: {[d.strftime('%Y-%m-%d') for d in dates[:5]]}")


In [ ]:
print(sample.pattern)

In [ ]:
dataset

In [ ]:
from tsl.data.datamodule import (SpatioTemporalDataModule,
                                 TemporalSplitter)
from tsl.data.preprocessing import StandardScaler

# Normalize data per time step (across nodes only) to maintain causality
# axis=(1,) normalizes across nodes (dimension 1) for each time step independently
# We need to fit on the full dataset because the scaler computes per-time-step statistics
# and the DataModule would otherwise fit only on the training slice (causing dimension mismatch)
scalers = {'target': StandardScaler(axis=(1,))}
# Split data sequentially:
#   |------------ dataset -----------|
#   |--- train ---|- val -|-- test --|
splitter = TemporalSplitter(val_len=0.1, test_len=0.05)

dm = SpatioTemporalDataModule(
    dataset=dataset,
    # scalers=scalers,
    splitter=splitter,
    batch_size=2,
)

In [ ]:
dm.setup()
print(dm)

## Custom STGNN

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.utils import to_dense_adj

from tsl.nn.blocks.encoders import RNN
from tsl.nn.layers import NodeEmbedding, GraphConv
from einops.layers.torch import Rearrange


class TimeThenSpaceModel(nn.Module):
    def __init__(self, input_size: int, n_nodes: int, horizon: int,
                 hidden_size: int = 32,
                 rnn_layers: int = 1):
        super(TimeThenSpaceModel, self).__init__()

        self.encoder = nn.Linear(input_size, hidden_size)
        self.node_embeddings = NodeEmbedding(n_nodes, hidden_size)

        self.time_nn = RNN(input_size=hidden_size,
                           hidden_size=hidden_size,
                           n_layers=rnn_layers,
                           cell='gru',
                           return_only_last_state=False)  # Return all states
        
        # Use GraphConv instead of DenseGraphConv for single graph processing
        self.space_nn = GraphConv(input_size=hidden_size,
                                   output_size=hidden_size,
                                   norm='mean',
                                   root_weight=True)

        self.decoder = nn.Linear(hidden_size, input_size * horizon)
        self.rearrange = Rearrange('b n (t f) -> b t n f', t=horizon)
        self.n_nodes = n_nodes

    def forward(self, x, edge_index, edge_weight):
        # x: [batch, time, nodes, features]
        batch_size, time_steps, n_nodes, n_features = x.shape
        
        x_enc = self.encoder(x)  # [batch, time, nodes, hidden_size]
        x_emb = x_enc + self.node_embeddings()  # add node-identifier embeddings
        
        # Temporal processing: RNN returns [batch, time, nodes, hidden_size]
        h_temporal = self.time_nn(x_emb)  # [batch, time, nodes, hidden_size]
        
        # Handle dynamic graphs: edge_index and edge_weight are lists
        if isinstance(edge_index, list):
            # Process each time step with its corresponding graph topology
            h_spatial_list = []
            for t in range(time_steps):
                # Get the graph for this time step
                if t < len(edge_index):
                    ei = edge_index[t]
                    ew = edge_weight[t] if isinstance(edge_weight, list) and t < len(edge_weight) else None
                else:
                    # Use last available graph if we run out
                    ei = edge_index[-1]
                    ew = edge_weight[-1] if isinstance(edge_weight, list) and len(edge_weight) > 0 else None
                
                # Get features for this time step: [batch, nodes, hidden_size]
                h_t = h_temporal[:, t, :, :]  # [batch, nodes, hidden_size]
                
                # Apply graph convolution with this time step's graph
                # GraphConv expects [batch, nodes, features] and returns [batch, nodes, features]
                h_spatial_t = self.space_nn(h_t, ei, ew)  # [batch, nodes, hidden_size]
                h_spatial_list.append(h_spatial_t)
            
            # Stack spatial features: [batch, time, nodes, hidden_size]
            h_spatial = torch.stack(h_spatial_list, dim=1)
        else:
            # Static graph case - process all time steps at once
            # We need to reshape to [batch * time, nodes, hidden_size] for GraphConv
            b, t, n, f = h_temporal.shape
            h_flat = h_temporal.view(b * t, n, f)  # [batch*time, nodes, hidden_size]
            h_spatial_flat = self.space_nn(h_flat, edge_index, edge_weight)  # [batch*time, nodes, hidden_size]
            h_spatial = h_spatial_flat.view(b, t, n, f)  # [batch, time, nodes, hidden_size]
        
        # Use the last time step's output for prediction
        h = h_spatial[:, -1, :, :]  # [batch, nodes, hidden_size]
        
        x_out = self.decoder(h)  # linear decoder: z=[b n f] -> x_out=[b n t⋅f]
        x_out = F.relu(x_out)  # Apply ReLU to ensure non-negative predictions
        x_horizon = self.rearrange(x_out)
        return x_horizon

In [ ]:
hidden_size = 32
rnn_layers = 1
gnn_kernel = 2

input_size = dataset.n_channels   # 1 channel
n_nodes = dataset.n_nodes         # 15 nodes
horizon = dataset.horizon         # 1 time step horizon

stgnn = TimeThenSpaceModel(input_size=input_size,
                           n_nodes=n_nodes,
                           horizon=horizon,
                           hidden_size=hidden_size,
                           rnn_layers=rnn_layers,
                           )

def print_model_size(model):
    tot = sum([p.numel() for p in model.parameters() if p.requires_grad])
    out = f"Number of model ({model.__class__.__name__}) parameters:{tot:10d}"
    print("=" * len(out))
    print(out)


print(stgnn)
print_model_size(stgnn)

In [ ]:
# from tsl.metrics.torch import MaskedMAE, MaskedMAPE
# from tsl.engines import Predictor

# loss_fn = MaskedMAE()

# metrics = {'mae': MaskedMAE(),
#            'mape': MaskedMAPE(),
#         #    'mae_at_15': MaskedMAE(at=2),  # '2' indicates the third time step,
#         #    'mae_at_30': MaskedMAE(at=5),
#         #    'mae_at_60': MaskedMAE(at=11)
#            }

# # setup predictor
# predictor = Predictor(
#     model=stgnn,                   # our initialized model
#     optim_class=torch.optim.Adam,  # specify optimizer to be used...
#     optim_kwargs={'lr': 0.001},    # ...and parameters for its initialization
#     loss_fn=loss_fn,               # which loss function to be used
#     metrics=metrics                # metrics to be logged during train/val/test
# )

from tsl.metrics.torch import MaskedMAE, MaskedMAPE, MaskedMetric
from tsl.engines import Predictor
import torch.nn.functional as F

class MaskedRMSLE(MaskedMetric):
    """Root Mean Squared Logarithmic Error Metric with masking support."""
    
    is_differentiable: bool = True
    higher_is_better: bool = False
    full_state_update: bool = False
    
    def __init__(self, mask_nans=False, mask_inf=False, at=None, **kwargs):
        def rmsle_fn(y_hat, y):
            log_pred = torch.log(y_hat + 1)
            log_actual = torch.log(y + 1)
            return (log_pred - log_actual) ** 2
        
        super(MaskedRMSLE, self).__init__(
            metric_fn=rmsle_fn,
            mask_nans=mask_nans,
            mask_inf=mask_inf,
            at=at,
            **kwargs
        )
    
    def forward(self, y_hat, y, mask=None):
        """Compute RMSLE directly for loss function usage."""
        # Handle time step selection if needed
        if self.at is not None and self.at != slice(None):
            y_hat = y_hat[:, self.at]
            y = y[:, self.at]
            if mask is not None:
                mask = mask[:, self.at]
        
        # Compute squared log error
        log_pred = torch.log(y_hat + 1)
        log_actual = torch.log(y + 1)
        squared_log_error = (log_pred - log_actual) ** 2
        
        # Apply mask if provided
        if mask is not None or self.mask_nans or self.mask_inf:
            if mask is None:
                mask = torch.ones_like(squared_log_error, dtype=torch.bool)
            else:
                mask = mask.bool()
            
            if self.mask_nans:
                mask = mask & ~torch.isnan(squared_log_error)
            if self.mask_inf:
                mask = mask & ~torch.isinf(squared_log_error)
            
            # Apply mask and compute mean
            masked_error = torch.where(mask, squared_log_error, torch.zeros_like(squared_log_error))
            mean_squared_log_error = masked_error.sum() / mask.sum().clamp(min=1)
        else:
            mean_squared_log_error = squared_log_error.mean()
        
        # Return RMSLE (sqrt of mean squared log error)
        return torch.sqrt(mean_squared_log_error)
    
    def compute(self):
        """Compute RMSLE from accumulated state for metric logging."""
        if self.numel > 0:
            value = self.value / self.numel
            return torch.sqrt(value)
        return self.value

# Use RMSLE as loss function
loss_fn = MaskedRMSLE()

metrics = {
    'rmsle': MaskedRMSLE(),
    'mae': MaskedMAE(),
    'mape': MaskedMAPE(),
}

# setup predictor
predictor = Predictor(
    model=stgnn,
    optim_class=torch.optim.Adam,
    optim_kwargs={'lr': 0.001},
    loss_fn=loss_fn,
    metrics=metrics
)

In [ ]:
from pytorch_lightning.loggers import TensorBoardLogger

logger = TensorBoardLogger(save_dir="logs", name="tsl_dev", version=0)

In [ ]:
%load_ext tensorboard
%tensorboard --logdir logs

In [ ]:
import pytorch_lightning as pl
from pytorch_lightning.callbacks import ModelCheckpoint

checkpoint_callback = ModelCheckpoint(
    dirpath='logs',
    save_top_k=1,
    monitor='val_mae',
    mode='min',
)

trainer = pl.Trainer(max_epochs=100,
                     logger=logger,
                     devices=1,
                     limit_train_batches=100,
                     callbacks=[checkpoint_callback])

trainer.fit(predictor, datamodule=dm)

In [ ]:
# predictor.load_model(checkpoint_callback.best_model_path)
predictor.freeze()
trainer.test(predictor, datamodule=dm);

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Set model to eval mode
predictor.eval()
predictor.freeze()

# Get predictions for all samples with proper time and node tracking
all_predictions = []
all_targets = []
all_split_indices = []  # train/val/test split
all_time_indices = []   # actual time step indices for each sample
all_sample_indices = [] # dataset sample indices
all_masks = []

# Get split indices from datamodule
# The datamodule splits have an 'indices' attribute that contains the sample indices
train_indices = dm.train_slice.indices.numpy() if hasattr(dm.train_slice, 'indices') else None
val_indices = dm.val_slice.indices.numpy() if hasattr(dm.val_slice, 'indices') else None
test_indices = dm.test_slice.indices.numpy() if hasattr(dm.test_slice, 'indices') else None

print("Collecting predictions with time and node tracking...")
print(f"  Train samples: {len(train_indices) if train_indices is not None else 0}")
print(f"  Val samples: {len(val_indices) if val_indices is not None else 0}")
print(f"  Test samples: {len(test_indices) if test_indices is not None else 0}")

# IMPORTANT: Iterate through dataset directly (not dataloader) to ensure correct order
# This ensures sample indices match exactly with the split indices
for split_name, split_sample_indices in [
    ('train', train_indices),
    ('val', val_indices),
    ('test', test_indices)
]:
    if split_sample_indices is None or len(split_sample_indices) == 0:
        continue
    
    print(f"Processing {split_name} split ({len(split_sample_indices)} samples)...")
    
    # Process samples in batches for efficiency, but maintain order
    batch_size = 32  # Process in batches but maintain sample order
    for batch_start in range(0, len(split_sample_indices), batch_size):
        batch_end = min(batch_start + batch_size, len(split_sample_indices))
        batch_sample_indices = split_sample_indices[batch_start:batch_end]
        
        # Get samples from dataset in order
        batch_data = [dataset[int(idx)] for idx in batch_sample_indices]
        
        # Create a batch from the samples
        from tsl.data import StaticBatch
        batch = StaticBatch.from_data_list(batch_data)
        
        with torch.no_grad():
            # Get predictions with postprocessing
            y_hat = predictor.predict_batch(batch, preprocess=False, postprocess=True)
            y_true = batch.y
            
            # Get mask if available
            mask = batch.get('mask', None)
            if mask is not None:
                all_masks.append(mask.cpu())
            
            # Store predictions and targets
            all_predictions.append(y_hat.cpu())
            all_targets.append(y_true.cpu())
            
            # Track sample indices and time step indices for this batch
            for i, sample_idx in enumerate(batch_sample_indices):
                all_sample_indices.append(int(sample_idx))
                
                # Get the actual time step index for the horizon of this sample
                horizon_idx = dataset.get_horizon_indices(int(sample_idx))
                # For horizon=1, horizon_idx is a single value
                if isinstance(horizon_idx, torch.Tensor):
                    if horizon_idx.numel() == 1:
                        time_step = horizon_idx.item()
                    else:
                        time_step = horizon_idx[0].item()  # Take first horizon step
                else:
                    time_step = int(horizon_idx)
                all_time_indices.append(time_step)
                
                all_split_indices.append(split_name)

# Concatenate all predictions and targets
predictions = torch.cat(all_predictions, dim=0)  # [n_samples, horizon, n_nodes, n_features]
targets = torch.cat(all_targets, dim=0)  # [n_samples, horizon, n_nodes, n_features]

# Verify mask if available
if len(all_masks) > 0:
    masks = torch.cat(all_masks, dim=0)  # [n_samples, horizon, n_nodes, n_features]
    print("Mask verification:")
    print(f"  Mask shape: {masks.shape}")
    print(f"  Mask dtype: {masks.dtype}")
    print(f"  Mask coverage (fraction of valid values): {masks.float().mean().item():.4f}")
    print(f"  Mask matches target shape: {masks.shape == targets.shape}")
else:
    print("Warning: No mask found in batches. MaskedMAE will use all values.")
    masks = None

# Convert to numpy arrays
all_time_indices = np.array(all_time_indices)
all_sample_indices = np.array(all_sample_indices)

# Verify time indices are valid
print(f"\nTime index verification:")
print(f"  Total samples: {len(all_time_indices)}")
print(f"  Valid time indices: {(all_time_indices >= 0).sum()}")
print(f"  Time index range: [{all_time_indices.min()}, {all_time_indices.max()}]")
print(f"  Dataset time steps: {dataset.n_steps}")

# Average across features only, keep nodes separate
# predictions: [n_samples, horizon, n_nodes, n_features] -> [n_samples, horizon, n_nodes]
predictions_nodes = predictions.mean(dim=3).squeeze(1).numpy()  # [n_samples, n_nodes]
targets_nodes = targets.mean(dim=3).squeeze(1).numpy()  # [n_samples, n_nodes]

# Use actual time step indices for plotting (not sample indices)
time_indices = all_time_indices

# Separate by split
train_mask = np.array([idx == 'train' for idx in all_split_indices])
val_mask = np.array([idx == 'val' for idx in all_split_indices])
test_mask = np.array([idx == 'test' for idx in all_split_indices])

# Get number of nodes
n_nodes = predictions_nodes.shape[1]

# Plot predictions vs actuals for each node
plt.figure(figsize=(16, 8))

# Plot each node's time series
# Use black for actual values, light red for predictions
actual_color = 'black'
pred_color = 'lightcoral'  # Light red color

for node_idx in range(n_nodes):
    alpha_actual = 0.7
    alpha_pred = 0.6
    
    # Plot training data
    if train_mask.any():
        train_indices = time_indices[train_mask]
        plt.plot(train_indices, targets_nodes[train_mask, node_idx], 
                color=actual_color, linestyle='-', linewidth=1.5, alpha=alpha_actual,
                label='Actual (Train)' if node_idx == 0 else '')
        plt.plot(train_indices, predictions_nodes[train_mask, node_idx], 
                color=pred_color, linestyle='-', linewidth=1.5, alpha=alpha_pred,
                label='Predicted (Train)' if node_idx == 0 else '')
    
    # Plot validation data
    if val_mask.any():
        val_indices = time_indices[val_mask]
        plt.plot(val_indices, targets_nodes[val_mask, node_idx], 
                color=actual_color, linestyle='-', linewidth=1.5, alpha=alpha_actual)
        plt.plot(val_indices, predictions_nodes[val_mask, node_idx], 
                color=pred_color, linestyle='-', linewidth=1.5, alpha=alpha_pred)
    
    # Plot test data
    if test_mask.any():
        test_indices = time_indices[test_mask]
        plt.plot(test_indices, targets_nodes[test_mask, node_idx], 
                color=actual_color, linestyle='-', linewidth=1.5, alpha=alpha_actual)
        plt.plot(test_indices, predictions_nodes[test_mask, node_idx], 
                color=pred_color, linestyle='-', linewidth=1.5, alpha=alpha_pred)

# Add vertical lines to separate train/val/test regions
if train_mask.any() and val_mask.any():
    train_end = time_indices[train_mask].max() + 0.5
    plt.axvline(x=train_end, color='gray', linestyle='--', linewidth=1, alpha=0.5, label='Train/Val Split')
if val_mask.any() and test_mask.any():
    val_end = time_indices[val_mask].max() + 0.5 if val_mask.any() else train_end
    plt.axvline(x=val_end, color='gray', linestyle='--', linewidth=1, alpha=0.5, label='Val/Test Split')

plt.xlabel('Time Step Index', fontsize=12)
plt.ylabel('Target Value (per node)', fontsize=12)
plt.title(f'Model Predictions vs Actual Values for All {n_nodes} Nodes', fontsize=14, fontweight='bold')
plt.legend(loc='upper left', fontsize=8, ncol=2)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Print summary statistics
print("\nPrediction Summary:")
print(f"  Total samples: {len(predictions_nodes)}")
print(f"  Number of nodes: {n_nodes}")
print(f"  Train samples: {train_mask.sum()}")
print(f"  Val samples: {val_mask.sum()}")
print(f"  Test samples: {test_mask.sum()}")
print(f"\nPrediction statistics:")
print(f"  Predictions min: {predictions_nodes.min():.4f}, max: {predictions_nodes.max():.4f}, mean: {predictions_nodes.mean():.4f}")
print(f"  Targets min: {targets_nodes.min():.4f}, max: {targets_nodes.max():.4f}, mean: {targets_nodes.mean():.4f}")
print(f"\nMean Absolute Error (averaged across nodes):")
if train_mask.any():
    train_mae = np.mean(np.abs(predictions_nodes[train_mask] - targets_nodes[train_mask]))
    print(f"  Train MAE: {train_mae:.4f}")
if val_mask.any():
    val_mae = np.mean(np.abs(predictions_nodes[val_mask] - targets_nodes[val_mask]))
    print(f"  Val MAE: {val_mae:.4f}")
if test_mask.any():
    test_mae = np.mean(np.abs(predictions_nodes[test_mask] - targets_nodes[test_mask]))
    print(f"  Test MAE: {test_mae:.4f}")

# Verify causality: Check that window indices come before horizon indices
print("\nCausality verification:")
print(f"  Dataset window: {dataset.window}")
print(f"  Dataset delay: {dataset.delay}")
print(f"  Dataset horizon: {dataset.horizon}")
print(f"  Horizon offset (window + delay): {dataset.horizon_offset}")
print(f"  ✓ Causality maintained: window indices [0, ..., {dataset.window-1}] come before horizon indices [{dataset.horizon_offset}, ..., {dataset.horizon_offset + dataset.horizon - 1}]")


In [ ]:
# Verification: Compare predictions to original dataframe
print("\n" + "="*60)
print("Alignment Verification:")
print("="*60)

# Get the original dataframe
df_original = dataset.dataframe()
if isinstance(df_original.columns, pd.MultiIndex):
    node_data_original = df_original.xs(0, level='channels', axis=1)
else:
    node_data_original = df_original

# For a few sample nodes and time steps, verify alignment
print("\nChecking alignment for a few samples:")
n_check = min(5, len(all_time_indices))
for i in range(n_check):
    time_idx = all_time_indices[i]
    sample_idx = all_sample_indices[i]
    split = all_split_indices[i]
    
    if time_idx >= 0 and time_idx < len(node_data_original):
        # Get original value for first node at this time step
        original_val = node_data_original.iloc[time_idx, 0]
        predicted_val = predictions_nodes[i, 0]
        target_val = targets_nodes[i, 0]
        
        print(f"  Sample {i} (split={split}, sample_idx={sample_idx}, time_idx={time_idx}):")
        print(f"    Original[time={time_idx}, node=0] = {original_val:.2f}")
        print(f"    Target[time={time_idx}, node=0] = {target_val:.2f}")
        print(f"    Predicted[time={time_idx}, node=0] = {predicted_val:.2f}")
        print(f"    Match: {'✓' if abs(original_val - target_val) < 1e-3 else '✗'} (diff={abs(original_val - target_val):.4f})")
    else:
        print(f"  Sample {i}: Invalid time_idx={time_idx}")

print("\nNote: Predictions should align with original data at the corresponding time steps.")
print("If there are mismatches, check that node order matches CBSA list order.")


In [ ]:
predictions_nodes.max()

In [ ]:
dataset.dataframe().max()

In [ ]:
import matplotlib.pyplot as plt

# Get the dataframe from the dataset
df = dataset.dataframe()

# The dataframe has MultiIndex columns: (nodes, channels)
# For single channel data, we can select the first channel or flatten
print(f"DataFrame shape: {df.shape}")
print(f"DataFrame columns (first 5): {df.columns[:5]}")
print(f"DataFrame index (first 5): {df.index[:5]}")

# If we have MultiIndex columns, extract node-level data
if isinstance(df.columns, pd.MultiIndex):
    # For single channel, select channel 0 for all nodes
    if dataset.n_channels == 1:
        # Select all nodes, channel 0
        node_data = df.xs(0, level='channels', axis=1)
    else:
        # If multiple channels, we'll plot the first channel
        node_data = df.xs(0, level='channels', axis=1)
else:
    node_data = df

print(f"\nNode data shape: {node_data.shape}")
print(f"Number of nodes: {node_data.shape[1]}")
print(f"Number of time steps: {node_data.shape[0]}")

# Plot time series for each node
plt.figure(figsize=(16, 10))

# Plot each node's time series
for node_idx in range(node_data.shape[1]):
    node_name = node_data.columns[node_idx] if hasattr(node_data.columns[node_idx], '__str__') else f'Node {node_idx}'
    plt.plot(node_data.index, node_data.iloc[:, node_idx], 
             alpha=0.6, linewidth=1, label=node_name if node_idx < 10 else '')  # Only label first 10 for clarity

plt.xlabel('Time Step', fontsize=12)
plt.ylabel('Target Value', fontsize=12)
plt.title(f'Time Series for All {node_data.shape[1]} Nodes', fontsize=14, fontweight='bold')
plt.legend(loc='upper left', fontsize=8, ncol=2)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Print summary statistics
print(f"\nSummary Statistics:")
print(f"  Total nodes: {node_data.shape[1]}")
print(f"  Total time steps: {node_data.shape[0]}")
print(f"  Min value across all nodes: {node_data.values.min():.2f}")
print(f"  Max value across all nodes: {node_data.values.max():.2f}")
print(f"  Mean value across all nodes: {node_data.values.mean():.2f}")
print(f"  Std value across all nodes: {node_data.values.std():.2f}")

